# Behavior Analysis for BMI Sessions

Clean version of the exploratory analysis in `first_pass_analysis.ipynb`.

The BMI task context from Hakim et al. is: the animal is rewarded for moving population neural activity along a selected decoder axis. The decoder produces a one-dimensional cursor, audiovisual feedback is given during active trials, and rewards occur when the cursor crosses the reward threshold. This notebook summarizes behavioral performance around that task: reward rates, hit rates, trial lengths, reward timing, remapping alignment, and optional cursor diagnostics.

This notebook should be safe to rerun for one animal by changing only the configuration cell below.

Run `%pip install -e .` once from this repo's notebook environment before importing `bci_analysis`.


In [ ]:
from pathlib import Path
import richfile as rfs

from bci_analysis.behavior import *

PROJECT_ROOT = Path.cwd()

register_richfile_types(rfs)


In [ ]:
# -------- user settings --------
MOUSE_NAME = "mouse_0403L"
SAMPLING_RATE = 30  # Hz

DATA_ROOT = Path(r"C:\Data\datasets\bmi_packets\bmi_packets")
OUTPUT_ROOT = PROJECT_ROOT / "output"

SLIDING_WINDOW_MIN = 2.5
SMOOTHING_WINDOW_SIZE = 20
FIRST_K_DAYS = 5
N_FIRST_DAYS = 6
K_REWARDS = 20

INCLUDE_CURSOR_DIAGNOSTICS = True
SHOW_FIGURES = True


In [ ]:
experiment_log_path = find_experiment_log_path(DATA_ROOT, PROJECT_ROOT)
experiment_tracking = load_experiment_tracking(experiment_log_path)

r = open_mouse_richfile(rfs, DATA_ROOT, MOUSE_NAME)

analysis = build_behavior_analysis(
    r,
    mouse_name=MOUSE_NAME,
    output_root=OUTPUT_ROOT,
    experiment_tracking=experiment_tracking,
    sampling_rate=SAMPLING_RATE,
    sliding_window_min=SLIDING_WINDOW_MIN,
    smoothing_window_size=SMOOTHING_WINDOW_SIZE,
)

print(f"Loaded {analysis.mouse_name}")
print(f"Days: {len(analysis.day_list)}")
print(f"Day zero: {analysis.phase.day_zero} (index {analysis.phase.day_zero_idx})")
print(f"First remapping day: {analysis.phase.first_remapping_day}")
print(f"End remapping day: {analysis.phase.end_remapping_phase}")
print(f"Output folder: {analysis.output_dir}")


In [ ]:
summary = make_daily_summary_table(analysis)
summary_path = analysis.output_dir / f"{analysis.mouse_name}_behavior_daily_summary.csv"
summary.to_csv(summary_path, index=False)

print(f"Saved daily summary: {summary_path}")
display(summary)


## Overview Plots

These reproduce the over-day behavioral summaries from the exploratory notebook: active/total reward rate, hit rate, mouse weight with successful trial length, and time to the first `K_REWARDS` active rewards.


In [ ]:
saved_paths = {}

saved_paths["reward_rate_over_days"] = plot_reward_rates_over_days(analysis, show=SHOW_FIGURES)
saved_paths["hit_rate_over_days"] = plot_hit_rate_over_days(analysis, show=SHOW_FIGURES)
saved_paths["weight_and_trial_length"] = plot_weight_and_trial_length(analysis, show=SHOW_FIGURES)
saved_paths[f"time_to_{K_REWARDS}_rewards"] = plot_time_to_k_rewards(
    analysis,
    k_rewards=K_REWARDS,
    show=SHOW_FIGURES,
)

saved_paths


## Reward Timing Across Sessions

The reward-rate traces use active rewards in sliding windows. Remapping days are aligned so the remapping start is placed at 10 minutes, matching the exploratory plots.


In [ ]:
saved_paths["reward_rate_progression"] = plot_reward_rate_progression(
    analysis,
    first_k_days=None,
    show=SHOW_FIGURES,
)
saved_paths[f"reward_rate_progression_first_{FIRST_K_DAYS}"] = plot_reward_rate_progression(
    analysis,
    first_k_days=FIRST_K_DAYS,
    show=SHOW_FIGURES,
)
saved_paths["reward_times_raster"] = plot_reward_times_raster(
    analysis,
    show=SHOW_FIGURES,
)
saved_paths["reward_times_raster_first_15_min"] = plot_reward_times_raster(
    analysis,
    xlim_min=15,
    suffix="_first_15_min",
    show=SHOW_FIGURES,
)

saved_paths


## Successful Trial Duration Distributions

These plots use the successful trial length as the reward-achievement time within each true BMI trial.


In [ ]:
saved_paths.update(
    plot_reward_time_distributions(
        analysis,
        n_first_days=N_FIRST_DAYS,
        show=SHOW_FIGURES,
    )
)

saved_paths


## Cursor Diagnostics

Optional checks comparing the logged WaveSurfer cursor, the paper-formula neural decoder reconstruction converted to tone frequency, and the logged `freqToOutput`. These are useful for sanity-checking the decoder implementation, but the core behavioral plots above do not depend on them.


In [ ]:
if INCLUDE_CURSOR_DIAGNOSTICS:
    saved_paths["cursor_diagnostic"] = plot_cursor_diagnostic(
        analysis,
        r,
        day_idx=analysis.phase.first_remapping_day_idx,
        start_time_min=9,
        duration_min=3,
        show=SHOW_FIGURES,
    )
    saved_paths["cursor_aligned_to_remapping"] = plot_cursor_aligned_to_remapping(
        analysis,
        r,
        k_days=9,
        show=SHOW_FIGURES,
    )

saved_paths


## Notes

Two obvious cleanup fixes are included here compared with the exploratory notebook:

- Remapping markers use `end_remapping_phase_idx` rather than the date string.
- Sliding reward-rate traces are reported in rewards/min by dividing counts by the window length in minutes.
